# [학습 정리] 관계형 DB 설계와 Django ORM 통합 마스터 (저자-도서 1:N)

## 1. 초기 환경 세팅 및 프로젝트 생성
가장 먼저 가상환경을 분리하고, 요구사항에 맞춘 프로젝트와 앱을 생성합니다. `.gitignore`는 반드시 가장 먼저 설정해야 합니다.

```bash
# 1. 가상환경 생성 및 활성화
python -m venv venv
source venv/Scripts/activate  # (Windows 기준)

# 2. Django 설치 및 프로젝트/앱 생성
pip install django
django-admin startproject libraries_management .
python manage.py startapp libraries

# ※ 생성 후 libraries_management/settings.py의 INSTALLED_APPS에 'libraries'를 반드시 추가해야 합니다.

## 2. [핵심 대조] Raw SQL vs Django ORM 비교
데이터베이스를 설계할 때, 과거의 방식(SQL)과 현재의 방식(ORM)이 어떻게 달라지는지 명확히 비교합니다.

* 방식 1: ORM을 쓰지 않았을 때 (Raw SQL 직접 작성)
과거에는 파이썬 코드 안에서 아래와 같이 길고 복잡한 SQL 텍스트를 직접 짜서 DB로 보내야 했습니다.

In [ ]:
-- 저자 테이블 생성
CREATE TABLE author (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(100) NOT NULL
);

-- 도서 테이블 생성 (1:N 관계 설정)
CREATE TABLE book (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title VARCHAR(100) NOT NULL,
    author_id INTEGER NOT NULL,
    -- 외래키(ForeignKey) 제약조건 설정
    FOREIGN KEY (author_id) REFERENCES author(id) ON DELETE CASCADE
);

* 방식 2: Django ORM을 사용했을 때 (현재 방식)
SQL을 전혀 몰라도, 파이썬의 클래스(Class) 문법만으로 DB 테이블을 똑같이 만들어냅니다. ForeignKey 단 한 줄이 위 SQL의 FOREIGN KEY 제약조건과 ON DELETE CASCADE를 완벽히 대체합니다.

In [ ]:
# libraries/models.py
from django.db import models

class Author(models.Model):
    name = models.CharField(max_length=100)

    # [매직 메서드] 관리자 페이지나 쉘에서 객체를 출력할 때 'Author object (1)' 대신 실제 '이름'을 보여주도록 설정
    def __str__(self):
        return self.name

class Book(models.Model):
    # 1:N 관계의 핵심: 도서(N)가 저자(1)를 참조합니다.
    author = models.ForeignKey(Author, on_delete=models.CASCADE)
    title = models.CharField(max_length=100)

    def __str__(self):
        return self.title

* **위 파이썬 클래스를 진짜 DB(SQLite)의 테이블로 번역하여 박아넣으려면 반드시 터미널에서 다음 두 명령어를 쳐야 합니다.
python manage.py makemigrations (설계도 작성)
python manage.py migrate (설계도를 DB에 실제 시공)**

## 3. 관리자 페이지 설정(Admin)

    *Django의 가장 강력한 기능 중 하나인 기본 관리자 페이지를 세팅합니다.

In [ ]:
# libraries/admin.py
from django.contrib import admin
from .models import Author, Book

# 관리자(admin) 사이트에서 Author와 Book 모델을 제어할 수 있도록 등록합니다.
admin.site.register(Author)
admin.site.register(Book)

* **실행 방법:**
    터미널에서 python manage.py createsuperuser로 최고 관리자 계정을 하나 만듭니다.
    서버를 켜고(python manage.py runserver) 브라우저에서 http://127.0.0.1:8000/admin/으로 접속하면, 저자를 2명 이상 자유롭게 생성해 볼 수 있습니다. (이때 모델에서 설정한 __str__ 덕분에 저자의 진짜 이름이 뜹니다.)

## 4. 저자 전체 목록 조회 (역참조의 활용)
이제 DB에 들어간 저자들의 목록과, 각 저자가 몇 권의 책을 썼는지 확인하는 페이지를 만듭니다.

* **View로직**

In [ ]:


# libraries/views.py
from django.shortcuts import render
from .models import Author

def author_list(request):
    # DB에서 모든 저자(Author) 데이터를 끌어옵니다. (SQL의 SELECT * FROM author 와 동일)
    authors = Author.objects.all()
    context = {
        'authors': authors,
    }
    return render(request, 'libraries/author_list.html', context)

* **Template 로직 (HTML)**
Author 모델 안에는 book이라는 필드가 없지만, 1:N 관계이기 때문에 장고가 몰래 쥐여준 역참조 매니저(book_set)를 활용하여 책의 수를 셉니다.

In [ ]:
<h1>저자 전체 목록</h1>
<hr>

<ul>
  {% for author in authors %}
    <li>
      <h3>저자명: {{ author.name }}</h3>
      <p>출간 도서 수: {{ author.book_set.count }}권</p>
    </li>
  {% empty %}
    <p>등록된 저자가 없습니다.</p>
  {% endfor %}
</ul>